# Cross-dataset MSN integration — SlideTags + NucSeq + HD (Handsaker) — Harmony

Simpler sibling of `01_integrate_HD_NucSeq_SlideTags_scArches.ipynb`: same data-prep pipeline
(cell-type selection, Ensembl-ID gene overlap, donor-ID harmonization across NucSeq/SlideTags,
sex/status harmonization), but integrates all **three datasets jointly with Harmony** instead of
training an scVI/scANVI reference and mapping onto it with scArches surgery. No reference/query
split needed -- Harmony corrects all three at once.

Runs on the **full** data by default (`SUBSET_RUN = False`) -- flip it on for a quick smoke test.

Unlike the scArches notebook, HD is **not** given a special extra cell-type restriction here --
all three datasets go through the exact same `CELL_TYPES_TO_INTEGRATE` filter, so there's no
apples-to-oranges population mismatch working against the integration.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import re
import itertools

import numpy as np
import pandas as pd
import scanpy as sc
import anndata
import matplotlib.pyplot as plt
import seaborn as sns

from utils import label_mapping_utils as lmu

print(f"scanpy: {sc.__version__}")


scanpy: 1.11.5


/tmp/ipykernel_15655/1800664065.py:17: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f"scanpy: {sc.__version__}")


## 1 · Config

In [2]:
BASE_PATH = "/home/gdallagl/myworkdir/XDP/data/XDP/cross_dataset_integration/MSN_harmony"
os.makedirs(BASE_PATH, exist_ok=True)

RAW_PATHS = {
    "HD":        "/home/gdallagl/myworkdir/XDP/data/HD_Handsaker/adata/HD_Handsaker_merged_labelled_zoned_VILLAGE_curated.h5ad",
    "nucseq":    "/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/adatas/Striatum/Striatum_combined_QC_mmc_ct-cured_zoned.h5ad",
    "slidetags": "/home/gdallagl/myworkdir/XDP/data/XDP/SlideTags_dataset/all_lib_adata_zoned_ct-cured_aligned_OLD.h5ad",
}

CELL_TYPE_COL   = "Group_name"   # used in Section 2 to choose which cells to integrate (coarse MapMyCells label)
TRAIN_LABEL_COL = "ct_for_deg"   # finer, project-curated subtype -- used for plotting/validation only here (no classifier)

# Quick smoke-test switch -- leave False for a real run
SUBSET_RUN = True
MAX_CELLS_PER_TYPE = 2000   # cap per TRAIN_LABEL_COL category, per dataset, when SUBSET_RUN is True

# Optional: restrict to a single disease status and/or sex. Both None = keep everything.
FILTER_STATUS = "Diseased"   # e.g. "Diseased" or "Healthy"
FILTER_SEX = None      # e.g. "Male" or "Female"

RAW_CONDITION_COL = {"HD": "Status", "nucseq": "condition", "slidetags": "condition"}
CONDITION_MAP = {
    "HD":        {"Case": "Diseased", "Control": "Healthy"},
    "nucseq":    {"XDP": "Diseased", "Control": "Healthy"},
    "slidetags": {"diseased": "Diseased", "healthy": "Healthy"},
}

# Technical pooling/library unit per dataset -> used as the `batch` column
BATCH_COL_MAP = {
    "HD":        "reaction",      # HD "Village" pooled reactions, coarsened from individual reactions (see Section 5)
    "nucseq":    "library",      # NucSeq multiplexed pool
    "slidetags": "Batch",        # SlideTags puck/capture area, joined in from a metadata csv (see Section 5)
}

# Donor column per dataset (raw, dataset-specific format)
DONOR_COL_MAP = {"HD": "DONOR", "nucseq": "donor_id", "slidetags": "donor_id"}

# Sex column per dataset -- collected differently across datasets, see Section 5.
# SlideTags/NucSeq are ~all male because XDP is X-linked; HD is a balanced HD cohort.
RAW_SEX_COL = {"HD": "Sex", "nucseq": "sex", "slidetags": "sex"}
SEX_MAP = {
    "HD":        {"Male": "Male", "Female": "Female"},
    "nucseq":    {"M": "Male", "F": "Female"},
    "slidetags": {"male": "Male", "female": "Female"},
}

MIN_BATCH_SIZE = 100   # drop batches with fewer cells than this (too few cells for a reliable batch estimate)
N_TOP_GENES = 3000

# Harmony config. hvg_batch_key must be a single column (scanpy requirement) with robustly-sized
# groups -- "dataset" (2-3 levels, thousands of cells each) is the safest choice, safer than any
# per-donor/per-library key. harmony_vars_use can be a list -- start with just ["dataset"] to
# isolate whether cross-study identity is what's failing to integrate, then add "donor_id_unified"
# on top once that's confirmed working (see notebook 01's troubleshooting history for why).
HVG_BATCH_KEY = "dataset"
HARMONY_VARS_USE = ["dataset"]
N_PCS = 50
MAX_ITER_HARMONY = 30
RESOLUTIONS = [0.1, 0.5, 0.07]
SEED = 42


## 2 · Choose which cells to integrate

Applied identically to **all three datasets** -- unlike the scArches notebook, there is no
per-dataset special-casing here, so whatever population exists on one side is expected to exist
on the other (a real precondition for any integration method to be able to merge them).

In [3]:
df_slidetags_batch = pd.read_csv("/home/gdallagl/myworkdir/XDP/utils/SlideTags_XDP_metadata/XDP Slidetags - CellBender Terra table - cb.csv")
df_slidetags_batch = df_slidetags_batch[["donor_id", "Batch"]]


In [4]:
CELL_TYPES_TO_INTEGRATE = [
    "STRd D1 Matrix MSN",
    "STRd D1 Striosome MSN",
    "STRd D2 Matrix MSN",
    "STRd D2 Striosome MSN",

    # "STRv D1 MSN",
    # "STRv D2 MSN",

    # "STRd D2 StrioMat Hybrid MSN",
    # "STR D1D2 Hybrid MSN",
    # "STRv D1 NUDAP MSN",
]

# other options you could swap in instead, e.g.:
# CELL_TYPES_TO_INTEGRATE = ["Microglia", "Macrophages"]
# CELL_TYPES_TO_INTEGRATE = ["Astrocyte"]


In [5]:
_backed = {name: sc.read_h5ad(p, backed="r") for name, p in RAW_PATHS.items()}

for name, a in _backed.items():
    vc = a.obs[CELL_TYPE_COL].value_counts()
    n_sel = vc.reindex(CELL_TYPES_TO_INTEGRATE).fillna(0).astype(int)
    print(f"--- {name} ({a.n_obs} cells total) ---")
    print(n_sel)
    missing = [ct for ct in CELL_TYPES_TO_INTEGRATE if ct not in vc.index or vc.get(ct, 0) == 0]
    if missing:
        print(f"  WARNING: not present in {name}: {missing}")
    print()


--- HD (581273 cells total) ---
Group_name
STRd D1 Matrix MSN       48558
STRd D1 Striosome MSN    19780
STRd D2 Matrix MSN       43363
STRd D2 Striosome MSN    14421
Name: count, dtype: int64

--- nucseq (231080 cells total) ---
Group_name
STRd D1 Matrix MSN       7896
STRd D1 Striosome MSN    5336
STRd D2 Matrix MSN       9321
STRd D2 Striosome MSN    4069
Name: count, dtype: int64

--- slidetags (537658 cells total) ---
Group_name
STRd D1 Matrix MSN       5835
STRd D1 Striosome MSN    9087
STRd D2 Matrix MSN       9129
STRd D2 Striosome MSN    6578
Name: count, dtype: int64



## 3 · Gene overlap across the three datasets

All three datasets use **Ensembl gene IDs as `var_names`**. Overlap is computed on `var_names`,
NOT on the `gene_symbol` column -- HD's `gene_symbol` annotation is older/different, so
symbol-based overlap with HD undercounts by ~10k genes even where the Ensembl IDs do match.

In [6]:
var_names_sets = {name: set(a.var_names) for name, a in _backed.items()}

for name, a in _backed.items():
    is_ensembl = a.var_names.str.match(r"^ENSG\d+").mean()
    print(f"{name}: var_names example={list(a.var_names[:2])}, {is_ensembl:.0%} look like Ensembl IDs")
    assert is_ensembl > 0.95, f"{name}: var_names doesn't look like Ensembl IDs -- check before trusting the overlap below"

print("\nPairwise overlap (Ensembl IDs, i.e. var_names):")
for a, b in itertools.combinations(var_names_sets, 2):
    print(f"  {a} ({len(var_names_sets[a])}) vs {b} ({len(var_names_sets[b])}): {len(var_names_sets[a] & var_names_sets[b])}")

COMMON_GENES = sorted(set.intersection(*var_names_sets.values()))
print(f"\nGenes shared by all three datasets (by Ensembl ID): {len(COMMON_GENES)}")


HD: var_names example=['ENSG00000121410', 'ENSG00000268895'], 100% look like Ensembl IDs
nucseq: var_names example=['ENSG00000223972', 'ENSG00000243485'], 96% look like Ensembl IDs
slidetags: var_names example=['ENSG00000290825', 'ENSG00000243485'], 100% look like Ensembl IDs

Pairwise overlap (Ensembl IDs, i.e. var_names):
  HD (36943) vs nucseq (38601): 30692
  HD (36943) vs slidetags (37464): 31763
  nucseq (38601) vs slidetags (37464): 35576

Genes shared by all three datasets (by Ensembl ID): 30560


In [7]:
# Symbol-based overlap, for comparison only -- DO NOT use this to subset
symbol_sets = {name: set(a.var["gene_symbol"].astype(str)) for name, a in _backed.items()}
print("Pairwise overlap (gene_symbol, i.e. gene NAMES -- comparison only):")
for a, b in itertools.combinations(symbol_sets, 2):
    print(f"  {a} vs {b}: {len(symbol_sets[a] & symbol_sets[b])}")
print("\n=> COMMON_GENES (used everywhere downstream) is the Ensembl-ID overlap above, not this one.")


Pairwise overlap (gene_symbol, i.e. gene NAMES -- comparison only):
  HD vs nucseq: 20738
  HD vs slidetags: 20971
  nucseq vs slidetags: 35056

=> COMMON_GENES (used everywhere downstream) is the Ensembl-ID overlap above, not this one.


## 4 · Donor ID harmonization

NucSeq and SlideTags are two modalities of the same XDP cohort, and several donors were profiled
in **both** -- normalize their donor IDs (differing separators/multiplex-pool suffixes) so shared
donors are identifiable. HD (Handsaker) is a separate cohort with a completely different, already-
consistent ID scheme (`S#####`), so it's excluded from normalization and checked for non-collision
instead.

In [8]:
def normalize_donor_id(x):
    """Normalize donor IDs across NucSeq/SlideTags: uppercase, unify separators,
    strip multiplexing-pool suffixes like 'CM', 'CF', 'CM2', 'CM6'."""
    x = str(x).strip().upper()
    if x in ("NAN", "UNASSIGNED", "NA", ""):
        return None
    x = x.replace("_", "-")
    m = re.match(r"^([A-Z]+)-?(\d+)-?(\d+)", x)
    if m:
        prefix, a, b = m.groups()
        return f"{prefix}-{a}-{b}"
    return x

NORMALIZE_DONOR_IDS = {"nucseq", "slidetags"}   # HD's S##### IDs are already consistent; normalizing would mangle them

donor_ids = {name: _backed[name].obs[DONOR_COL_MAP[name]].astype(str) for name in RAW_PATHS}
donor_id_unified = {
    name: (ids.map(normalize_donor_id) if name in NORMALIZE_DONOR_IDS else ids)
    for name, ids in donor_ids.items()
}

shared_nucseq_slidetags = set(donor_id_unified["slidetags"].dropna()) & set(donor_id_unified["nucseq"].dropna())
print(f"SlideTags donors: {donor_id_unified['slidetags'].nunique()}")
print(f"NucSeq donors: {donor_id_unified['nucseq'].nunique()}")
print(f"Donors shared between SlideTags and NucSeq: {len(shared_nucseq_slidetags)}")
print(sorted(shared_nucseq_slidetags))

other_ids = set(donor_id_unified["nucseq"].dropna()) | set(donor_id_unified["slidetags"].dropna())
assert not (set(donor_id_unified["HD"].dropna().unique()) & other_ids), "HD donor IDs unexpectedly collide with SCF/PCMC namespace"


SlideTags donors: 9
NucSeq donors: 47
Donors shared between SlideTags and NucSeq: 9
['SCF-18-003', 'SCF-18-006', 'SCF-19-014', 'SCF-20-024', 'SCF-20-025', 'SCF-20-026', 'SCF-22-043', 'SCF-22-057', 'SCF-23-068']


## 5 · Subset each dataset: chosen cell types x common genes, load to memory

In [ ]:
def load_and_prepare(name):
    print(name)
    a = sc.read_h5ad(RAW_PATHS[name], backed="r")

    # Strip everything we don't need BEFORE to_memory() -- combined fancy-indexing on a backed
    # h5ad (rows + columns at once) and materializing every layer/obsm/obsp for the full file
    # is what OOMs on files this large.
    for layer in list(a.layers.keys()):
        del a.layers[layer]
    for key in list(a.obsp.keys()):
        del a.obsp[key]
    for key in list(a.obsm.keys()):
        if not (name == "slidetags" and key == "spatial"):
            del a.obsm[key]   # keep SlideTags' aligned spatial coords just long enough to copy into obs below

    mask = a.obs[CELL_TYPE_COL].isin(CELL_TYPES_TO_INTEGRATE)
    a = a[mask].to_memory().copy()   # subset rows first, while still backed
    a = a[:, COMMON_GENES].copy()    # subset columns once already in memory

    if name == "slidetags":
        # obs["x"]/["y"] are per-puck RAW coordinates (independent local frames); obsm["spatial"]
        # is the cross-puck ALIGNED coordinates -- keep the aligned ones as obs columns so they
        # survive anndata.concat with HD/nucseq (obsm keys don't survive concat unless present
        # in every object being merged; obs columns do)
        a.obs["spatial_x"] = a.obsm["spatial"][:, 0]
        a.obs["spatial_y"] = a.obsm["spatial"][:, 1]
        del a.obsm["spatial"]
        a.obs["Batch"] = a.obs["donor_id"].map(df_slidetags_batch.set_index("donor_id")["Batch"].to_dict())

    if name == "HD":
        a.obs["village"] = a.obs["reaction"].str.split("_").str[0]

    a.obs["dataset"] = name
    a.obs["donor_id_unified"] = donor_id_unified[name].loc[a.obs_names].values
    a.obs["batch"] = name + "__" + a.obs[BATCH_COL_MAP[name]].astype(str)
    a.obs[CELL_TYPE_COL] = a.obs[CELL_TYPE_COL].astype(str)
    a.obs[TRAIN_LABEL_COL] = a.obs[TRAIN_LABEL_COL].astype(str)

    # harmonize sex -- categories must stay within {"Male","Female"}; unmapped/missing dropped
    a.obs["sex"] = a.obs[RAW_SEX_COL[name]].astype(str).str.strip().map(SEX_MAP[name])
    n_missing_sex = a.obs["sex"].isna().sum()
    if n_missing_sex:
        print(f"  dropping {n_missing_sex} cells with unknown/missing sex")
    a = a[a.obs["sex"].notna()].copy()

    # harmonize disease status (human-readable "Diseased"/"Healthy", also used by FILTER_STATUS)
    a.obs["is_diseased"] = a.obs[RAW_CONDITION_COL[name]].astype(str).str.strip().map(CONDITION_MAP[name])
    assert a.obs["is_diseased"].notna().all(), f"{name}: unmapped condition value(s)"

    # status/sex filters run BEFORE SUBSET_RUN's cap -- otherwise the cap draws a random sample
    # of Diseased+Healthy (or Male+Female) cells per category, and the *subsequent* filter then
    # discards a very different fraction of that capped sample per dataset (HD is ~50% Diseased,
    # NucSeq ~63%, SlideTags ~88%), silently shrinking whichever dataset has the lowest match
    # rate far more than the others -- even if it had the most raw cells to begin with.
    if FILTER_STATUS is not None:
        a = a[a.obs["is_diseased"] == FILTER_STATUS].copy()
        print(f"  FILTER_STATUS='{FILTER_STATUS}': kept {a.n_obs} cells")
    if FILTER_SEX is not None:
        a = a[a.obs["sex"] == FILTER_SEX].copy()
        print(f"  FILTER_SEX='{FILTER_SEX}': kept {a.n_obs} cells")

    if SUBSET_RUN:
        keep_names = []
        for ct, idx in a.obs.groupby(TRAIN_LABEL_COL, observed=True).groups.items():
            n = min(len(idx), MAX_CELLS_PER_TYPE)
            keep_names.append(pd.Series(idx).sample(n=n, random_state=SEED).values)
        a = a[np.concatenate(keep_names)].copy()
        print(f"  SUBSET_RUN: capped to {MAX_CELLS_PER_TYPE} cells per '{TRAIN_LABEL_COL}' -> {a.n_obs} cells")

    a.layers["counts"] = a.layers["counts"].copy() if "counts" in a.layers else a.X.copy()
    a.X = a.layers["counts"].copy()

    # drop batches with too few cells to estimate a batch effect
    batch_counts = a.obs["batch"].value_counts()
    a = a[a.obs["batch"].isin(batch_counts[batch_counts >= MIN_BATCH_SIZE].index)].copy()

    a.var_names_make_unique()
    print(f"{name}: {a.shape} after cell-type + gene filtering (+ tiny-batch removal)")
    return a


adatas = {name: load_and_prepare(name) for name in RAW_PATHS}
del _backed


HD
  FILTER_STATUS='Diseased': kept 28524 cells
  SUBSET_RUN: capped to 2000 cells per 'ct_for_deg' -> 9371 cells
HD: (8439, 30560) after cell-type + gene filtering (+ tiny-batch removal)
nucseq
  FILTER_STATUS='Diseased': kept 7442 cells
  SUBSET_RUN: capped to 2000 cells per 'ct_for_deg' -> 7442 cells
nucseq: (7442, 30560) after cell-type + gene filtering (+ tiny-batch removal)
slidetags


In [ ]:
# QC: per-donor cell counts for the chosen cell type(s), per dataset
for name, a in adatas.items():
    print(f"--- {name}: cells per donor (chosen cell types) ---")
    print(a.obs.groupby("donor_id_unified", observed=True).size().sort_values(ascending=False).describe())
    print()


## 6 · Harmony integration (all three datasets jointly)

No reference/query split -- Harmony corrects HD + NucSeq + SlideTags simultaneously.

In [ ]:
def run_harmony_integration(adata, hvg_batch_key, harmony_vars_use, n_top_genes=N_TOP_GENES,
                             n_pcs=N_PCS, max_iter_harmony=MAX_ITER_HARMONY,
                             resolutions=RESOLUTIONS, seed=SEED):
    """Same steps as lmu.preprocess_common_pipeline(..., integrate_with_harmony=True), but with
    the HVG batch_key (must be a single column) decoupled from Harmony's vars_use (may be a list
    of columns) -- lmu's version reuses one `batch_key` argument for both, which breaks if you
    want Harmony to correct for more than one covariate.
    """
    lmu.set_global_seeds(seed)

    adata.X = adata.layers["counts"].copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.layers["log1p_norm"] = adata.X.copy()

    try:
        sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes, flavor="seurat_v3",
                                     layer="counts", batch_key=hvg_batch_key)
    except ValueError as e:
        # seurat_v3 fits a per-group loess curve; a group with too few/degenerate cells can
        # produce a singular fit -- fall back to no batch_key rather than crash
        print(f"WARNING: HVG with batch_key='{hvg_batch_key}' failed ({e}), retrying without batch_key")
        sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes, flavor="seurat_v3",
                                     layer="counts", batch_key=None)

    adata_hvg = adata[:, adata.var["highly_variable"]].copy()
    sc.pp.scale(adata_hvg, max_value=10)
    sc.tl.pca(adata_hvg, svd_solver="randomized", n_comps=n_pcs, random_state=seed)
    adata.obsm["X_pca"] = adata_hvg.obsm["X_pca"]
    adata.uns["pca"] = adata_hvg.uns["pca"]
    del adata_hvg

    import harmonypy
    ho = harmonypy.run_harmony(adata.obsm["X_pca"], adata.obs, vars_use=harmony_vars_use, max_iter_harmony=max_iter_harmony)
    Z = ho.Z_corr
    adata.obsm["X_pca_harmony"] = Z.T if Z.shape[0] != adata.n_obs else Z

    sc.pp.neighbors(adata, use_rep="X_pca_harmony", n_pcs=n_pcs, random_state=seed)
    sc.tl.umap(adata, random_state=seed)

    print("Computing Leiden clusters...")
    for r in resolutions:
        sc.tl.leiden(adata, resolution=r, key_added=f"leiden_{r}_harmony",
                     flavor="igraph", n_iterations=2, random_state=seed)

    adata.X = adata.layers["counts"].copy()  # restore raw counts in X
    return adata


In [ ]:
adata_harmony = anndata.concat(
    {name: a for name, a in adatas.items()},
    join="inner",
    label="dataset",
    index_unique="-",
    merge="first",   # keeps var columns like gene_symbol -- "first" not "same" because HD's
                     # gene_symbol annotation sometimes disagrees with NucSeq/SlideTags for the
                     # same Ensembl ID (older annotation vintage); "same" would just drop it
)
print(adata_harmony)


In [ ]:
adata_harmony.obs["dataset"].value_counts()

In [ ]:
adata_harmony = run_harmony_integration(
    adata_harmony,
    hvg_batch_key=HVG_BATCH_KEY,
    harmony_vars_use=HARMONY_VARS_USE,
)


## 7 · Visualize

In [ ]:
sc.pl.umap(adata_harmony, color=["dataset", TRAIN_LABEL_COL, "batch", "sex", "is_diseased", "leiden_0.5_harmony"],
           ncols=2, frameon=False, palette=lmu.custom_palette)

In [ ]:
sc.settings.set_figure_params(figsize=(10, 8))
sc.pl.umap(adata_harmony, color=["dataset", TRAIN_LABEL_COL, "batch", "sex", "is_diseased", "leiden_0.5_harmony"],
           ncols=2, frameon=False, palette=lmu.custom_palette)


In [ ]:
sc.pl.umap(adata_harmony, color="ct_for_deg",groups=["-Phase_D"], frameon=False, title="Disease status (per-study definition)", palette=lmu.custom_palette, size=50)


In [ ]:
# Sanity check for the donor harmonization in Section 4: pick a couple of donors profiled in
# BOTH NucSeq and SlideTags and confirm their cells land in the same region
example_shared_donors = sorted(shared_nucseq_slidetags)[:3]
adata_harmony.obs["highlight_shared_donor"] = np.where(
    adata_harmony.obs["donor_id_unified"].isin(example_shared_donors),
    adata_harmony.obs["donor_id_unified"] + " (" + adata_harmony.obs["dataset"] + ")",
    "other",
)
sc.pl.umap(
    adata_harmony, color="highlight_shared_donor",
    groups=[d for d in adata_harmony.obs["highlight_shared_donor"].unique() if d != "other"],
    frameon=False, title="Same donor, two modalities",
)


## Save outputs

In [ ]:
out_path = f"{BASE_PATH}/adata_harmony_integrated.h5ad"
adata_harmony.write(out_path)
print("Saved to", out_path)
